In [1]:
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions

#### Antioxidant protein dataset preprocessing and metadata generation

This script processes antioxidant protein data from multiple sheets within a single source file, assigns labels based on class information, and generates a structured dataset along with metadata.

- Overview
    - Task: Antioxidant protein classification dataset preparation
    - Source: Zhang et al. dataset
    - Input: Excel file with training and independent datasets containing protein sequences and class labels and metadata Excel file
    - Output: Processed dataset (CSV) and metadata file (JSON)
- Process:
    - Read protein sequences from multiple sheets in an Excel file
    - Assign labels based on class values (antioxidant vs non-antioxidant)
    - Concatenate training and independent datasets
    - Remove unnecessary columns and standardize column names
    - Load and organize source metadata
    - Export data

- Auxiliary variables

In [2]:
path_export = "../../processed_dataset"
path_input = "../../raw_dataset"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Zhang et al"

- Reading data

In [3]:
df_training = pd.read_excel(f"{path_input}/{name_source}/pone.0163274.s001.xlsx", sheet_name="Training Dataset")
df_training["label"] = np.where(
    df_training["Class"] == "Non-antioxidant",
    0,
    1
)
df_training.head()

,Class,Acc_id,Sequence,label
0,Non-antioxidant,O93829,MQHGIKRVKLSEEAKRLKLEKDQIKIKNYRQLTDEIFELRANENYS...,0
1,Non-antioxidant,Q93JL5,MARDIAAPPVPTNHQELISWVNEIAELTQPDAVVWCDGSEAEYERL...,0
2,Non-antioxidant,Q63415,MPPRAPPAPGPRPPPRAAGRHGLSPLAPRPWRWLLLLALPAVCSAL...,0
3,Non-antioxidant,O28216,MSSEKEVQEKIATLQILQEEAEALQRRLMELEILENEYRKTLETLE...,0
4,Non-antioxidant,Q59214,MTIKKIAVLTSGGDSQGMNAAVRAVVRSGLFYGLEVYGIQRGYQGL...,0


In [4]:
df_independent = pd.read_excel(f"{path_input}/{name_source}/pone.0163274.s001.xlsx", sheet_name="Independent Testing Dataset")
df_independent["label"] = np.where(
    df_independent["Class"] == "Non-antioxidant",
    0,
    1
)
df_independent.head()

,Class,Acc_id,Sequence,label
0,Non-antioxidant,Q5A343,MRSPSLAVAATTVLGLFSSSALAYYGNTTTVALTTTEFVTTCPYPT...,0
1,Non-antioxidant,Q5RDG4,MRLRRLALFPGVALLLAAARLAAASDVLELTDDNFESRISDTGSAG...,0
2,Non-antioxidant,Q5DRB4,MAPPQRHPQRSEQVLLLTLLGTLWGAAAAQIRYSIPEELEKGSFVG...,0
3,Non-antioxidant,B0TQX6,MSELSDIRREYTLGELHSEDVPNDPMDLFNAWLEVVRDSQIQDPTA...,0
4,Non-antioxidant,Q2FTH1,MAGNAAVGVLALQGDVSEHISAFESAIQNLGLNIPVVPVRKAEQIL...,0


- Concatenating data

In [5]:
df_concat = pd.concat([df_training, df_independent], axis=0, ignore_index=True)
df_concat.drop(columns=["Class", "Acc_id"], inplace=True)
df_concat.rename(columns={"Sequence": "sequence"}, inplace=True)
df_concat.head()

,sequence,label
0,MQHGIKRVKLSEEAKRLKLEKDQIKIKNYRQLTDEIFELRANENYS...,0
1,MARDIAAPPVPTNHQELISWVNEIAELTQPDAVVWCDGSEAEYERL...,0
2,MPPRAPPAPGPRPPPRAAGRHGLSPLAPRPWRWLLLLALPAVCSAL...,0
3,MSSEKEVQEKIATLQILQEEAEALQRRLMELEILENEYRKTLETLE...,0
4,MTIKKIAVLTSGGDSQGMNAAVRAVVRSGLFYGLEVYGIQRGYQGL...,0


In [6]:
df_concat.shape, df_concat["sequence"].unique().shape

((666, 2), (666,))

In [7]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
56,pone.0163274.s001,Zhang et al,Dataset,Static,Creative Commons Attribution License,No,2016,2016-09-23,2025-09-07,xlsx,"Sequence, UniProt ID",Enzyme/protein classification,Antioxidant,Sampling from Swiss-Prot,Sampling from Swiss-Prot,Supplementary Material,https://journals.plos.org/plosone/article?id=1...,No information


In [8]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'pone.0163274.s001',
 'name source': 'Zhang et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution License',
 'reports constant updates': 'No',
 'year of publication': 2016,
 'last update date': Timestamp('2016-09-23 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Sampling from Swiss-Prot',
 'obtaining positive dataset': 'Sampling from Swiss-Prot',
 'repository or server': 'Supplementary Material',
 'publication': 'https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0163274',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-17 22:34:56'}

In [9]:
dict_metadata['number_of_records'] = df_concat.shape[0]
dict_metadata['number_of_collected_sequences'] = df_concat.shape[0]
dict_metadata['number_of_unique_sequences'] = df_concat.shape[0]
dict_metadata['positive_examples'] = df_concat[df_concat["label"] == 1].shape[0]
dict_metadata['negative_examples'] = df_concat[df_concat["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = 0
dict_metadata

{'name dataset': 'pone.0163274.s001',
 'name source': 'Zhang et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution License',
 'reports constant updates': 'No',
 'year of publication': 2016,
 'last update date': Timestamp('2016-09-23 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'xlsx',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'Sampling from Swiss-Prot',
 'obtaining positive dataset': 'Sampling from Swiss-Prot',
 'repository or server': 'Supplementary Material',
 'publication': 'https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0163274',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-17 22:34:56',
 'number_of_records': 666,
 'number_of_collected_sequences': 666,
 'number_of_unique_sequences': 666,
 'positive_examples': 174,
 'neg

- Export data

In [10]:
UtilsFunctions.make_directory(f"{path_export}/{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}/{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
df_concat.to_csv(f"{path_export}/{name_task}/{name_source}/processed_data.csv", index=False)